# 数组创建与形状

学习目标：创建 NumPy 数组，理解形状与轴，掌握常用创建方法和逐元素运算。

前置知识：Python 变量、列表与元组、索引、模块导入和基本算术。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的数据，后续单元沿用首次导入的 np。

## 1 ndarray 与数组创建

ndarray 是 NumPy 的多维数组对象。np.array() 可以把列表转换为数组；np 是 numpy 的常用导入别名。

Python 列表可以容纳不同种类的对象，普通数值数组则使用统一的元素类型。数组用 shape 描述形状，用 dtype 描述元素类型。

In [1]:
import numpy as np

values = [2.0, 4.0, 6.0]
numbers = np.array(values)

print(type(values), type(numbers))  # 分别为 list、numpy.ndarray。
print(numbers)  # 预期：[2. 4. 6.]
print(numbers.dtype)  # 预期：float64，数组元素为浮点数。

<class 'list'> <class 'numpy.ndarray'>
[2. 4. 6.]
float64


## 2 数组形状与轴

### 2.1 维数、形状与元素数量

数组的维数表示轴的数量，形状表示各轴长度。下面三个属性直接通过点号读取，无需加括号。

| 属性 | 中文名称／含义 |
| --- | --- |
| ndim | 维数，即轴的数量 |
| shape | 形状，各轴长度组成的元组 |
| size | 元素总数 |

列表可以创建一维数组，规则的嵌套列表可以创建二维或更高维数组。一维数组只有一个轴，没有单独的行轴和列轴。

In [2]:
vector = np.array([1, 2, 3])
measurements = np.array([[20.0, 21.0, 22.0], [19.0, 20.0, 21.0]])

print(vector.ndim, vector.shape, vector.size)  # 预期：1 (3,) 3
print(measurements.ndim, measurements.shape, measurements.size)  # 预期：2 (2, 3) 6

1 (3,) 3
2 (2, 3) 6


### 2.2 轴的编号与含义

轴从 0 开始编号。二维数组常用行、列描述两个轴；形状 (2, 3) 表示两行三列。

继续使用上面的 measurements，约定行表示观测、列表示传感器。轴 0 的长度是观测次数，轴 1 的长度是传感器数量；这些实际含义由使用者约定。shape[0] 和 shape[1] 读取的是形状元组中的长度。

In [3]:
print(measurements)
# 两行分别是两次观测，三列分别是三个传感器。
print("观测次数：", measurements.shape[0])  # 2。
print("传感器数量：", measurements.shape[1])  # 3。

[[20. 21. 22.]
 [19. 20. 21.]]
观测次数： 2
传感器数量： 3


## 3 逐元素运算

### 3.1 算术运算

同形数组可以按对应位置计算。例如，三个测量读数分别减去对应基准，可以一次得到三个偏差，无需逐项写循环。

同形数组的 +、-、*、/ 分别进行逐元素加、减、乘、除，结果形状不变。其中 * 是逐元素乘法，不是矩阵乘法。

下面两个输入的形状均为 (3,)，普通算术表达式生成新结果，不直接修改输入。

In [4]:
readings = np.array([2.0, 4.0, 6.0])
baseline = np.array([1.0, 2.0, 3.0])

print(readings - baseline)  # 偏差为 [1. 2. 3.]，每个读数减去对应基准。
print(readings + baseline)  # 预期：[3. 6. 9.]
print(readings * baseline)  # 预期：[ 2.  8. 18.]
print(readings / baseline)  # 预期：[2. 2. 2.]；本例除数均不为零。
print(readings)  # 输入仍为 [2. 4. 6.]。

[1. 2. 3.]
[3. 6. 9.]
[ 2.  8. 18.]
[2. 2. 2.]
[2. 4. 6.]


### 3.2 比较运算

同形数组的比较也逐元素进行，得到同形的布尔数组。每个 True 或 False 表示对应位置是否满足条件。

下面 readings 和 thresholds 的形状均为 (3,)，分别保存三个读数与对应阈值。

In [5]:
readings = np.array([20.0, 21.5, 23.0])
thresholds = np.array([21.0, 21.0, 23.0])

above = readings > thresholds
at_threshold = readings == thresholds

print(above)  # [False  True False]：只有第二个温度高于阈值。
print(at_threshold)  # [False False  True]：第三个温度等于阈值。
print(above.shape, above.dtype)  # (3,) bool。

[False  True False]
[False False  True]
(3,) bool


## 4 多维数组与特殊形状

### 4.1 三维数组

两批数据，每批三次观测，每次四个传感器读数，可以组织成形状 (2, 3, 4) 的数组。三个轴依次表示批次、观测和传感器。读嵌套列表前，先把每一批看成一张三行四列的小表：沿轴 0 换表，沿轴 1 换行，沿轴 2 换列。

![两个三行四列的数据块分别是批次 0 和 1，三个轴依次表示批次、观测和传感器。](image/illustration/01-01-array-axes.svg)

图中块与行列展示的是数据分组，不是内存排布；各轴的业务含义由本例约定。三维表示有三个轴，不表示有三行。

下面用与图中相同的数值构造 batches。先数出最外层有几个块、每块有几行、每行有几个值，再用 shape、ndim 和 size 核对。

In [6]:
batches = np.array([
    # 第一批：三次观测，每次四个传感器读数。
    [[11, 12, 13, 14], [21, 22, 23, 24], [31, 32, 33, 34]],
    # 第二批：结构相同，数值用来区分批次。
    [[111, 112, 113, 114], [121, 122, 123, 124], [131, 132, 133, 134]],
])

print(batches)  # 两个分块，每块三行四列，与嵌套列表的分组对应。
print(batches.shape)  # 预期：(2, 3, 4)，依次为批次、观测、传感器数量。
print(batches.ndim, batches.size)  # 预期：3 24，共 2 × 3 × 4 个元素。

[[[ 11  12  13  14]
  [ 21  22  23  24]
  [ 31  32  33  34]]

 [[111 112 113 114]
  [121 122 123 124]
  [131 132 133 134]]]
(2, 3, 4)
3 24


### 4.2 零维数组与空数组

零维数组没有轴，但包含一个元素；空数组没有元素，size 为 0，两者不同。

np.array(7) 的形状是 ()，np.array([7]) 的形状是 (1,)。空列表得到形状 (0,) 的一维数组；形状 (0, 3) 则有两个轴，其中一个轴的长度为 0。

下面用 np.empty((0, 3)) 创建指定形状的空数组；empty 的初始化行为在第 6.2 节说明。

In [7]:
scalar_array = np.array(7)
length_one = np.array([7])
empty_vector = np.array([])
empty_table = np.empty((0, 3))

print("零维：", scalar_array.shape, scalar_array.size)  # () 1。
print("长度为 1：", length_one.shape, length_one.size)  # (1,) 1。
print("空的一维：", empty_vector.shape, empty_vector.size)  # (0,) 0。
print("空的二维：", empty_table.shape, empty_table.ndim, empty_table.size)
# (0, 3) 2 0：轴的数量仍是 2，但元素总数为 0。

零维： () 1
长度为 1： (1,) 1
空的一维： (0,) 0
空的二维： (0, 3) 2 0


## 5 array 与 asarray

np.array() 和 np.asarray() 都能接收列表、元组等输入。默认情况下，array() 复制数组数据，asarray() 只在需要时复制。

对于 dtype 和存储顺序均符合要求的普通 ndarray，asarray() 可以返回原对象，不能把它当作获取独立副本的方法。下面用 is 判断是否为同一个对象。

In [8]:
source = np.array([2.0, 4.0, 6.0])
converted = np.asarray(source)
copied = np.array(source)
from_tuple = np.asarray((8.0, 10.0))

print(from_tuple)  # [8. 10.]，元组也可以转换成数组。
print(converted is source)  # True：这个输入不需要复制。
print(copied is source)  # False：默认 np.array() 创建了新数组。
print(copied)  # 数值仍然是 [2. 4. 6.]。

[ 8. 10.]
True
False
[2. 4. 6.]


## 6 数组初始化

### 6.1 zeros、ones 与 full

指定形状和初值，可以创建固定大小的数组。下面的形状 (2, 3) 表示两行三列。

| 函数 | 中文名称／含义 |
| --- | --- |
| np.zeros() | 创建全零数组 |
| np.ones() | 创建全一数组 |
| np.full() | 创建指定初值的数组 |

zeros() 和 ones() 默认使用 float64；full() 默认根据填充值确定类型。dtype 参数可以指定类型，下面用 float 表示浮点数。

In [9]:
zeros = np.zeros((2, 3))
ones = np.ones((2, 3))
baseline = np.full((2, 3), 20.0, dtype=float)

print(zeros)  # 两行三列，全部为 0.0。
print(ones)  # 两行三列，全部为 1.0。
print(baseline)  # 两行三列，全部为 20.0。
print(baseline.shape, baseline.dtype)  # (2, 3) 和 float64。

[[0. 0. 0.]
 [0. 0. 0.]]
[[1. 1. 1.]
 [1. 1. 1.]]
[[20. 20. 20.]
 [20. 20. 20.]]
(2, 3) float64


### 6.2 empty 与未初始化内容

np.empty() 按指定形状分配数组，但不初始化数值元素。使用前应先为每个位置赋值，不能把偶然出现的零当作确定初值。

fill() 把数组的所有元素设为指定值。形状和元素数量可以直接查看，数组内容则在填充后读取。

In [10]:
workspace = np.empty((2, 3))

print(workspace.shape, workspace.size)  # (2, 3) 6，并不是空数组。
workspace.fill(0.0)
print(workspace)  # 完成全部赋值后，可以确定六个元素都是 0.0。

(2, 3) 6
[[0. 0. 0.]
 [0. 0. 0.]]


## 7 等间隔序列

### 7.1 arange：按步长生成

np.arange(start, stop, step) 中，start 是起点，stop 是停止边界，step 是步长。使用整数参数时，包含起点、不包含停止边界。

只提供一个参数时，它表示 stop，start 默认为 0，step 默认为 1。

In [11]:
sample_numbers = np.arange(5)
even_numbers = np.arange(2, 10, 2)

print(sample_numbers)  # [0 1 2 3 4]。
print(even_numbers)  # [2 4 6 8]，不包含 10。

[0 1 2 3 4]
[2 4 6 8]


### 7.2 linspace：按点数生成

np.linspace(start, stop, num) 中，start 和 stop 是区间端点，num 是点数，必须是非负整数。下面取 5 个点，默认包含两个端点；endpoint=False 排除终点，并相应调整间距。

未指定 dtype 时，即使端点是整数，linspace() 也会采用浮点类型。

In [12]:
including_stop = np.linspace(0, 1, num=5)
excluding_stop = np.linspace(0, 1, num=5, endpoint=False)

print(including_stop)  # [0.   0.25 0.5  0.75 1.  ]。
print(excluding_stop)  # [0.  0.2 0.4 0.6 0.8]。
print(including_stop.size, excluding_stop.size)  # 两者都是 5 个点。

[0.   0.25 0.5  0.75 1.  ]
[0.  0.2 0.4 0.6 0.8]
5 5


### 7.3 浮点步长的边界

arange() 使用浮点参数时，舍入误差可能影响结果长度和终点附近的取值。下面计算得到的停止边界并不恰好等于十进制的 0.3。

需要固定点数或明确包含终点时，使用 linspace()；生成的小数仍可能存在浮点表示误差。

In [13]:
floating_stop = 0.1 * 3
floating_steps = np.arange(0.0, floating_stop, 0.1)

print(floating_stop)  # 0.30000000000000004。
print(floating_steps)  # 本例有 0.0、0.1、0.2、0.3 四个值。
print(floating_steps.size)  # 4，不是从直觉中的半开区间猜出的 3。
print(np.linspace(0.0, 0.3, num=4))  # 用点数明确要求生成 4 个值。

0.30000000000000004
[0.  0.1 0.2 0.3]
4
[0.  0.1 0.2 0.3]


## 8 综合应用：整理测量数据

把三个时刻、两个传感器的模拟温度组织成二维数组，约定行表示时刻、列表示传感器，单位为摄氏度。

创建同形的 20.0 °C 基准数组，计算各位置的偏差，并判断是否高于基准。

In [14]:
measurement_rows = [[20.0, 21.0], [21.0, 22.0], [22.0, 23.0]]
readings_table = np.asarray(measurement_rows, dtype=float)
reference_table = np.full((3, 2), 20.0)
deviations = readings_table - reference_table
above_reference = readings_table > reference_table

print(readings_table.shape, readings_table.size)  # (3, 2) 6。
print(deviations)  # 三行依次为 [0, 1]、[1, 2]、[2, 3] °C。
print(above_reference)  # 只有左上角为 False，其余位置为 True。
print(deviations.shape)  # 运算后仍是 (3, 2)。

(3, 2) 6
[[0. 1.]
 [1. 2.]
 [2. 3.]]
[[False  True]
 [ True  True]
 [ True  True]]
(3, 2)


## 9 选学：模板数组与对角数组

### 9.1 模板数组

以下函数默认沿用输入数组的形状与 dtype，初值由函数决定，不复制原数组的数值。

| 函数 | 中文名称／含义 |
| --- | --- |
| np.zeros_like() | 参照输入创建全零数组 |
| np.ones_like() | 参照输入创建全一数组 |
| np.full_like() | 参照输入创建指定初值的数组 |
| np.empty_like() | 参照输入创建未初始化的数值数组 |

full_like() 不会仅因填充值是小数就自动改用浮点类型，下面使用浮点模板。empty_like() 的数值元素同样需要先赋值再读取。

In [15]:
template = np.array([[1.0, 2.0], [3.0, 4.0]])

print(np.zeros_like(template))  # 形状 (2, 2)，全为 0.0。
print(np.ones_like(template))  # 形状 (2, 2)，全为 1.0。
print(np.full_like(template, 0.5))  # 形状 (2, 2)，全为 0.5。

temporary = np.empty_like(template)
temporary.fill(2.0)
print(temporary)  # 先填充后读取，四个位置都是 2.0。

[[0. 0.]
 [0. 0.]]
[[1. 1.]
 [1. 1.]]
[[0.5 0.5]
 [0.5 0.5]]
[[2. 2.]
 [2. 2.]]


### 9.2 对角数组

主对角线从二维数组的左上角延伸到右下角。np.eye() 默认创建主对角线为 1、其余位置为 0 的二维数组；只指定行数时，列数与行数相同。

np.diag() 接收一维输入时构造对角数组，接收二维输入时提取对角线。下面均使用默认的主对角线。

In [16]:
identity = np.eye(3)
diagonal_table = np.diag([2.0, 3.0, 4.0])

print(identity)  # 3 × 3，主对角线为 1.0，其余位置为 0.0。
print(diagonal_table)  # 主对角线依次为 2.0、3.0、4.0。
print(np.diag(diagonal_table))  # [2. 3. 4.]。

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[[2. 0. 0.]
 [0. 3. 0.]
 [0. 0. 4.]]
[2. 3. 4.]


## 本章小结

（1）ndarray 用统一的 dtype 描述元素类型；ndim、shape、size 分别表示轴数、各轴长度和元素总数。

（2）零维数组含一个元素，空数组没有元素；空数组仍可以有多个轴。

（3）array() 默认复制数据，asarray() 可能返回原数组；empty() 创建的数值元素需要先赋值再使用。

（4）arange() 按步长生成序列，linspace() 按点数生成序列；浮点参数需要留意舍入。

（5）同形数组按对应位置进行算术和比较。运行前应能判断结果的形状，并解释每个轴的含义。

## 练习

（1）把下面的读数创建为数组，约定行表示观测、列表示传感器。打印维数、形状和元素总数，再减去同形的 19.0 基准数组。

In [17]:
readings = [[18.0, 20.0], [19.0, 21.0], [20.0, 22.0]]

# 在此创建数组与同形基准，计算并打印偏差。
# 提示：full() 可以创建指定形状和初值的数组。
# 检查：输入为二维，形状 (3, 2)，共 6 个元素；首行偏差为 -1.0、1.0。

（2）分别为以下需求选择合适的创建方法，在代码注释中说明选择理由，再生成数组并检查结果。

① 从 2 开始，每次增加 3，生成小于 14 的整数。

② 在 0～2 区间生成 5 个等间隔值，包含两个端点；再保持点数不变，改为排除终点。

说明两个需求分别固定了什么条件，以及排除终点后哪个条件会改变。

In [18]:
start = 2
stop = 14
step = 3
point_count = 5

# 需求①：在此说明方法选择理由，生成并打印数组。
# 检查：结果为 [2, 5, 8, 11]，相邻差为 3，不包含 14。

# 需求②：在此说明方法选择理由，生成 0～2 区间的两个数组。
# 检查：点数均为 5；包含端点时末值为 2.0，排除终点时末值小于 2.0。
# 比较两个数组的相邻间距，解释保持点数不变时排除终点的影响。

（3）先预测下面三个数组的 ndim、shape 和 size，再运行核对。另创建形状 (2, 2) 的 empty 数组，填充为 3.0 后打印。

In [19]:
scalar_array = np.array(5)
single_item = np.array([5])
empty_array = np.empty((2, 0))

# 先记录预测，再比较维数、各轴长度和元素数量。
print(scalar_array.ndim, scalar_array.shape, scalar_array.size)
print(single_item.ndim, single_item.shape, single_item.size)
print(empty_array.ndim, empty_array.shape, empty_array.size)

# 在此创建另一个数组，使用 fill() 完成全部赋值后再打印。
# 检查：形状为 (2, 2)，四个元素均为 3.0；不读取未初始化值作为答案。

0 () 1
1 (1,) 1
2 (2, 0) 0


### 重点练习提示

对应第（2）题。先独立完成，再按需要查看提示。

（1）先区分固定步长与固定点数，两者不是同一个条件。

（2）画出包含两个端点的 5 个点，数一数间隔；排除终点但仍保留 5 个点时，再数一次。

### 重点练习参考解析

对应第（2）题。

需求①固定步长，使用 arange(2, 14, 3)，得到 [2, 5, 8, 11]。需求②固定点数，使用 linspace(0, 2, 5)，得到 [0, 0.5, 1, 1.5, 2]；改为 endpoint=False 后得到 [0, 0.4, 0.8, 1.2, 1.6]。

包含终点时，4 个间隔覆盖长度 2，间距为 0.5；排除终点时按 5 个间隔划分，间距为 0.4。两份结果都有 5 个点。不能把排除终点理解为从原结果删掉末项，那样只剩 4 个点。

## 参考与引用来源

本章新增示意图由 CMYK Labs 原创，依据下表对应概念与本章教学输入绘制；示意图不作为实际运行截图或数学证明。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | 三维分组示意：[Absolute basics — What is an array? / Array fundamentals / Array attributes](https://numpy.org/doc/2.5/user/absolute_beginners.html)，多维数组与轴、shape 的含义； 创建与形状：[ndarray](https://numpy.org/doc/2.5/reference/arrays.ndarray.html) 的定义、Array attributes 与零维说明；[array](https://numpy.org/doc/2.5/reference/generated/numpy.array.html) 的 object、dtype、copy 参数；[asarray](https://numpy.org/doc/2.5/reference/generated/numpy.asarray.html) 的 copy 条件及已有数组示例。 初始化：[zeros](https://numpy.org/doc/2.5/reference/generated/numpy.zeros.html)、[ones](https://numpy.org/doc/2.5/reference/generated/numpy.ones.html)、[full](https://numpy.org/doc/2.5/reference/generated/numpy.full.html) 的形状、dtype 与初值；[empty](https://numpy.org/doc/2.5/reference/generated/numpy.empty.html) 的 Notes；[ndarray.fill](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.fill.html) 的全元素赋值。 序列与运算：[arange](https://numpy.org/doc/2.5/reference/generated/numpy.arange.html) 的区间、步长与 Warning；[linspace](https://numpy.org/doc/2.5/reference/generated/numpy.linspace.html) 的 num、endpoint 和 dtype；[Quickstart — Basic operations](https://numpy.org/doc/2.5/user/quickstart.html#basic-operations) 的逐元素算术、比较与乘法区别。 选学：[Array creation routines](https://numpy.org/doc/2.5/reference/routines.array-creation.html) 的 Ones and zeros；[full_like](https://numpy.org/doc/2.5/reference/generated/numpy.full_like.html) 的模板 dtype 与示例；[eye](https://numpy.org/doc/2.5/reference/generated/numpy.eye.html) 的 N、M；[diag](https://numpy.org/doc/2.5/reference/generated/numpy.diag.html) 的一维／二维输入与 k 参数。 |
| Python 官方文档（Python 3.12） | [An Informal Introduction to Python — Lists](https://docs.python.org/3.12/tutorial/introduction.html#lists)：列表可以包含不同类型的元素。 |